In [ ]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load the three CSV files using fs.folders
high_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")

# ============================================================
# CELL 6.1: PREPARE LABELED DATA
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# =====================
# CONFIGURATION
# =====================
CONFIG = {
    # Sampling limits (balance with labeled data)
    "sampling": {
        "unlabeled_multiplier": 3,  # Max unlabeled = labeled_size * 3
        "pseudo_multiplier": 10      # Max pseudo = labeled_size * 10
    }
}

print(f"\n{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"\nPseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"]
if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].sample(
        n=max_pseudo, random_state=42
    )
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled['is_pseudo'] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"]
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)}")
print(f"  Pseudo:      {len(df_pseudo_sampled)}")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)}")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# ============================================================

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([
    df_labeled,
    df_pseudo_sampled
], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([
    df_labeled,
    df_unlabeled_sampled
], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([
    df_labeled,
    df_pseudo_sampled,
    df_unlabeled_sampled
], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """
    Split data into train/val, using stratification if possible.
    Only stratify on labeled data (exclude UNLABELED from stratification).
    """
    # Separate labeled and unlabeled data
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()
    
    # Check if stratified split is possible for labeled data
    if len(labeled_data) > 0:
        topic_counts = labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                stratify=labeled_data['label'],
                random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
        
        # Add unlabeled data to training set only (not validation)
        if len(unlabeled_data) > 0:
            train_data = pd.concat([train_labeled, unlabeled_data], ignore_index=True)
            print(f"      Added {len(unlabeled_data)} unlabeled to training")
        else:
            train_data = train_labeled
        
        val_data = val_labeled
    else:
        # Edge case: only unlabeled data (shouldn't happen but handle it)
        train_data = data
        val_data = data.head(0)  # Empty validation set
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("TRAIN/VAL SPLIT SUMMARY")
print(f"{'='*60}")
print(f"\nOption 1 (Labeled only):")
print(f"  Training:   {len(train_opt1):>6} examples")
print(f"  Validation: {len(val_opt1):>6} examples")

print(f"\nOption 2 (Labeled + Pseudo):")
print(f"  Training:   {len(train_opt2):>6} examples")
print(f"  Validation: {len(val_opt2):>6} examples")

print(f"\nOption 3 (Labeled + Unlabeled):")
print(f"  Training:   {len(train_opt3):>6} examples")
print(f"  Validation: {len(val_opt3):>6} examples")

print(f"\nOption 4 (All) ⭐ RECOMMENDED:")
print(f"  Training:   {len(train_opt4):>6} examples")
print(f"  Validation: {len(val_opt4):>6} examples")

# =====================
# SAVE DATA
# =====================

print(f"\n{'='*60}")
print("SAVING DATA")
print(f"{'='*60}")

# Save label mapping
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
print(f"✓ Saved label mapping")

# Save all training and validation sets
fs.save_data(train_opt1, "train_data_option1_labeled_only", "Model_finetuning", "csv")
fs.save_data(val_opt1, "val_data_option1", "Model_finetuning", "csv")

fs.save_data(train_opt2, "train_data_option2_with_pseudo", "Model_finetuning", "csv")
fs.save_data(val_opt2, "val_data_option2", "Model_finetuning", "csv")

fs.save_data(train_opt3, "train_data_option3_with_unlabeled", "Model_finetuning", "csv")
fs.save_data(val_opt3, "val_data_option3", "Model_finetuning", "csv")

fs.save_data(train_opt4, "train_data_option4_all", "Model_finetuning", "csv")
fs.save_data(val_opt4, "val_data_option4", "Model_finetuning", "csv")

print(f"✓ Saved all training and validation sets")

# Save checkpoint
fs.save_config("checkpoint6_training_prep")
print(f"✓ Checkpoint saved")

print(f"\n{'='*60}")
print("✓ DATA PREPARATION COMPLETE")
print(f"{'='*60}")
print(f"\nReady for model training!")
print(f"Choose one of the training options (1-4) for your model.")